## Setup

Run this notebook from a local clone of the [svm-gmu](https://github.com/SushrutGaikwad/svm-gmu) repository. From the repo root, install the project and launch Jupyter with [uv](https://docs.astral.sh/uv/):

```bash
uv sync
uv run jupyter lab
```

The next cell puts the repo's `src/` and `experiments/` folders on the import path, so `svm_gmu`, the shared `_common` helper, and the `_realworld` module import from your local clone. Rendering uses your local LaTeX (`pdflatex`).

In [ ]:
import sys
from pathlib import Path


def _add_repo_to_path():
    """Put the repo's experiments/ and src/ folders on sys.path.

    Locates the svm-gmu repository by searching upward from the working
    directory, so the notebook runs from any local clone regardless of where
    Jupyter was started.
    """
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "experiments" / "_common.py").exists():
            exp, src = base / "experiments", base / "src"
        elif (base / "_common.py").exists():
            exp, src = base, base.parent / "src"
        else:
            continue
        for path in (str(exp), str(src)):
            if path not in sys.path:
                sys.path.insert(0, path)
        return
    raise RuntimeError(
        "Could not find the svm-gmu repository. Run this notebook from a local "
        "clone (for example, `uv run jupyter lab` from the repo root)."
    )


_add_repo_to_path()

import _common as C
import _realworld as RW

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
C.configure_pgf()
GRAPHICS_DIR = C.GRAPHICS_DIR

FORCE_RECOMPUTE = False
CONFIG = dict(
    digit_pos=4, digit_neg=9, master_seed=C.MASTER_SEED, n_seeds=30,
    aug_mode="bimodal", jitter=6.0, augment_test=True, test_aug_per=10,
    rot_ladder=[15.0, 30.0, 45.0], train_ladder=[25, 80, 200],
    fixed_R=30.0, fixed_n_train=80, n_test=200, n_aug=100, max_shift=1.0,
    pca_dim=20, k_anchors=5, n_per_anchor=40, lam_grid=[0.01],
    n_folds=2, svm_kwargs=dict(max_iter=500, batch_size=64), cov_type="diag",
)

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
images = mnist.data.astype(np.float64) / 255.0
labels = mnist.target.astype(int)

In [ ]:
import pickle
from pathlib import Path
# Pickle is safe here: the cache is written and read by this same notebook,
# stored in a local git-ignored .cache/ directory. No external sources.
cache = Path(C.__file__).resolve().parent / ".cache" / "realworld_mnist_30seed.pkl"
if cache.exists() and not FORCE_RECOMPUTE:
    res = pickle.loads(cache.read_bytes())
else:
    res = RW.run_mnist_experiment(images, labels, CONFIG)
    cache.parent.mkdir(exist_ok=True)
    cache.write_bytes(pickle.dumps(res))

In [ ]:
_MODEL_KEYS = ["B0", "B1", "M0", "M1", "M2"]
_MODEL_LABELS = {
    "B0": "LSVM (point)", "B1": "LSVM-iso", "M0": "SVM-GSU",
    "M1": "SVM-GMU (structural)", "M2": "SVM-GMU (EM)",
}

# Fixed operating point: fixed_R, fixed_n_train
op = res["rot_sweep"].get(CONFIG["fixed_R"])
if op is None:
    # Fall back to closest key if fixed_R is not in rot_ladder
    op = res["rot_sweep"][min(res["rot_sweep"], key=lambda k: abs(k - CONFIG["fixed_R"]))]

print(f"Results at R={CONFIG['fixed_R']} deg, n_train={CONFIG['fixed_n_train']} per class")
print()
header = f"{'Model':<24s}  {'Acc (med, IQR)':<22s}  {'F1 (med, IQR)':<22s}  {'AUC (med, IQR)':<22s}  {'AP (med, IQR)':<22s}"
print(header)
print("-" * len(header))
def fmt(triple):
    m, lo, hi = triple
    return f"{m:.3f} [{lo:.3f}, {hi:.3f}]"

for mkey in _MODEL_KEYS:
    row = op[mkey]
    print(f"{_MODEL_LABELS[mkey]:<24s}  {fmt(row['accuracy']):<22s}  {fmt(row['f1']):<22s}  {fmt(row['auc']):<22s}  {fmt(row['ap']):<22s}")

sig = res["significance"]
print()
print("Significance (SVM-GMU EM vs SVM-GSU):")
print(f"  Wilcoxon p = {sig['paired_seed']['wilcoxon_p']:.4g}")
print(f"  Paired t   = {sig['paired_seed']['ttest_p']:.4g}")
print(f"  McNemar p (median) = {sig['mcnemar_p_median']:.4g}")
print(f"  BIC component count (median) = {sig['bic_counts_median']:.2f}")

In [ ]:
fig1 = RW.make_sweep_figure(res["rot_sweep"], xlabel=r"Bimodal rotation magnitude (deg)", title="Accuracy vs bimodal rotation magnitude")
fig1.savefig(GRAPHICS_DIR / "realworld_mnist_30seed_rotation.pgf", bbox_inches="tight")
fig2 = RW.make_sweep_figure(res["train_sweep"], xlabel="Training images per class", title="Accuracy vs training size")
fig2.savefig(GRAPHICS_DIR / "realworld_mnist_30seed_trainsize.pgf", bbox_inches="tight")
fig3 = RW.plot_mnist_2d_panel(images, labels, CONFIG, seed=C.MASTER_SEED)
fig3.savefig(GRAPHICS_DIR / "realworld_mnist_30seed_panel.pgf", bbox_inches="tight")